# Arduino Programs from This Chat

This notebook collects every code-based question from the chat in notebook form.
The code is kept as close as possible to the original and only corrected where needed for obvious compile-time issues.


# 1) Measure Distance Using Ultrasonic Sensor and Make an LED Blink

## Question
Write an Arduino program to measure distance using an ultrasonic sensor and blink an LED when the object comes near.


In [ ]:
#define trigPin 9
#define echoPin 10
#define ledPin 13

long duration;
int distance;

void setup() {
  pinMode(trigPin, OUTPUT);
  pinMode(echoPin, INPUT);
  pinMode(ledPin, OUTPUT);

  Serial.begin(9600);
}

void loop() {

  digitalWrite(trigPin, LOW);
  delayMicroseconds(2);

  digitalWrite(trigPin, HIGH);
  delayMicroseconds(10);

  digitalWrite(trigPin, LOW);

  duration = pulseIn(echoPin, HIGH);

  distance = duration * 0.034 / 2;

  Serial.print("Distance: ");
  Serial.print(distance);
  Serial.println(" cm");

  if (distance < 20) {
    digitalWrite(ledPin, HIGH);
    delay(200);

    digitalWrite(ledPin, LOW);
    delay(200);
  }
  else {
    digitalWrite(ledPin, LOW);
  }

  delay(100);
}


# 2) LDR to Vary Intensity of LED Using Arduino

## Question
Write an Arduino program using LDR sensor to control LED based on light intensity.


In [ ]:
const int ldrPin = A0;
const int ledPin = 7;

int threshold = 500;

void setup() {
  pinMode(ledPin, OUTPUT);
  Serial.begin(9600);
}

void loop() {

  int ldrValue = analogRead(ldrPin);

  Serial.println(ldrValue);

  if (ldrValue > threshold) {
    digitalWrite(ledPin, HIGH);
  }
  else {
    digitalWrite(ledPin, LOW);
  }

  delay(500);
}


# 3) Detect the Vibration of Object Using Arduino

## Question
Write an Arduino program to detect vibration using vibration sensor, LED and buzzer.


In [ ]:
const int vibPin = 2;
const int ledPin = 7;
const int buzzPin = 8;

void setup() {

  pinMode(vibPin, INPUT);
  pinMode(ledPin, OUTPUT);
  pinMode(buzzPin, OUTPUT);

  Serial.begin(9600);
}

void loop() {

  int vibState = digitalRead(vibPin);

  if (vibState == LOW) {

    Serial.println("Vibration Detected");

    digitalWrite(ledPin, HIGH);
    digitalWrite(buzzPin, HIGH);

    delay(500);
  }
  else {

    Serial.println("Vibration Not Detected");

    digitalWrite(ledPin, LOW);
    digitalWrite(buzzPin, LOW);
  }

  delay(500);
}


# 4) Temperature Notification Using Arduino and DHT11 Sensor

## Question
Write an Arduino program to measure temperature and humidity using DHT11 sensor and upload data to ThingSpeak.


In [ ]:
#include <WiFiS3.h>
#include <DHT.h>

char ssid[] = "your_ssid";
char pass[] = "your_password";

String apiKey = "API_KEY";

const char* server = "api.thingspeak.com";

#define DHTPIN 2
#define DHTTYPE DHT11

DHT dht(DHTPIN, DHTTYPE);

WiFiClient client;

float tempThreshold = 30.0;
float humThreshold = 70.0;

void setup() {

  Serial.begin(9600);

  dht.begin();

  Serial.print("Connecting to WiFi");
  WiFi.begin(ssid, pass);

  int maxAttempts = 30;
  while (WiFi.status() != WL_CONNECTED && maxAttempts > 0) {
    Serial.print(".");
    delay(3000);
    maxAttempts--;
  }

  if (WiFi.status() == WL_CONNECTED) {
    Serial.println("\nConnected to WiFi");
  }
  else {
    Serial.println("\nFailed to connect to WiFi");
  }
}

void loop() {

  float temp = dht.readTemperature();
  float hum = dht.readHumidity();

  if (isnan(temp) || isnan(hum)) {
    Serial.println("Failed to read from DHT sensor");
    return;
  }

  Serial.print("Temperature: ");
  Serial.print(temp);

  Serial.print(" °C  Humidity: ");
  Serial.print(hum);

  Serial.println("%");

  if (client.connect(server, 80)) {

    String url = "/update?api_key=" + apiKey +
                 "&field1=" + String(temp) +
                 "&field2=" + String(hum);

    client.print(String("GET " ) + url +
                 " HTTP/1.1\r\n" +
                 "Host: " + server + "\r\n" +
                 "Connection: close\r\n\r\n");

    Serial.println("Data sent to ThingSpeak");
  }

  client.stop();

  if (temp > tempThreshold) {
    Serial.println("High Temperature Alert");
  }

  if (hum > humThreshold) {
    Serial.println("High Humidity Alert");
  }

  delay(2000);
}


# 5) Sense the Available Networks Using Arduino BLE

## Question
Write an Arduino BLE program to control an LED using Bluetooth Low Energy communication and nRF Connect app.


In [ ]:
#include <ArduinoBLE.h>

const int ledPin = 13;

BLEService ledService("12345678-1234-1234-1234-1234567890ab");

BLEByteCharacteristic ledCharacteristic(
  "abcd1234-5678-1234-5678-1234567890ab",
  BLERead | BLEWrite
);

void setup() {

  Serial.begin(9600);

  pinMode(ledPin, OUTPUT);

  if (!BLE.begin()) {

    Serial.println("Failed to start BLE");

    while (1);
  }

  BLE.setLocalName("LED_Control_On");

  BLE.setAdvertisedService(ledService);

  ledService.addCharacteristic(ledCharacteristic);

  BLE.addService(ledService);

  ledCharacteristic.writeValue((byte)0);

  BLE.advertise();

  Serial.println("BLE LED Control Ready");
}

void loop() {

  BLEDevice central = BLE.central();

  if (central) {

    Serial.print("Connected to: " );
    Serial.println(central.address());

    while (central.connected()) {

      if (ledCharacteristic.written()) {

        byte value = ledCharacteristic.value();

        if (value == '1') {

          digitalWrite(ledPin, HIGH);
          Serial.println("LED ON");
        }
        else if (value == '0') {

          digitalWrite(ledPin, LOW);
          Serial.println("LED OFF");
        }
      }
    }

    Serial.println("Disconnected");
  }
}


# 6) Connect with the Available WiFi Using Arduino

## Question
Write an Arduino program to connect Arduino Uno R4 WiFi to a wireless network and display the connection status and IP address using Serial Monitor.


In [ ]:
#include <WiFiS3.h>

const char* ssid = "wifi_name";
const char* password = "password";

void setup() {

  Serial.begin(115200);

  unsigned long startMillis = millis();

  while (!Serial && millis() - startMillis < 5000);

  Serial.println("Serial is working");

  Serial.println("Connecting to WiFi");
  Serial.println(ssid);

  WiFi.begin(ssid, password);

  int maxAttempts = 30;

  while (WiFi.status() != WL_CONNECTED && maxAttempts > 0) {

    Serial.print(".");

    delay(500);

    maxAttempts--;
  }

  if (WiFi.status() == WL_CONNECTED) {

    Serial.println("\nConnected to WiFi");

    Serial.print("IP Address: " );

    Serial.println(WiFi.localIP());
  }
  else {

    Serial.println("\nFailed to connect to WiFi");
  }
}

void loop() {

  if (WiFi.status() == WL_CONNECTED) {

    Serial.println("Connected IP Address:");

    Serial.println(WiFi.localIP());
  }
  else {

    Serial.println("Not Connected to WiFi");
  }

  delay(5000);
}
